In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/evan/Projects/llm-zoomcamp-2026-code/module_02_vector_search/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8184.40it/s]


In [3]:
q0 = "Can I still join the course after the start date?"
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the proram, can I still enroll?"
q3 = "How to install Docker on Windows?"

In [4]:
v0 = model.encode(q0)
v1 = model.encode(q1)
v2 = model.encode(q2)
v3 = model.encode(q3)

In [5]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [6]:
for v in [v0, v1, v2, v3]:
    print(v.dot(dv)) 

0.3233239
0.39572877
0.39020044
0.019730594


## Embedding Our Dataset

In [7]:
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [ ]:
# We concatenate ALL our questions and answers into a list of lists 
texts = [document['question'] + ' ' + document['answer'] for document in documents]

In [23]:
# for example, the question and answer from document 10 above is now one long string

print(f"question: {documents[10].get('question')}")
print(f"answer: {documents[10].get('answer')}")
texts[10]

question: Course: How many hours per week am I expected to spend on this course?
answer: It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.

You can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.


'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [15]:
len(texts)

1346

In [16]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)


len(vectors)

100%|██████████| 27/27 [00:11<00:00,  2.40it/s]


1346

## Implement Vector Search

In [30]:
import numpy as np 

#Conver the vectors into an numpy array for lightening fast processing
X = np.array(vectors)

In [31]:
#Still the same shape as above: we have the exact same number of documents
len(X)

1346

In [34]:
#Now we compute the similiarity from our first question above: 
print(q1)
scores = X.dot(v1)

I just discovered the course, can I still join?


In [36]:
#np.argmax finds the highest score. 
idx = np.argmax(scores)

#the code below returns the index for the highest score
idx, scores[idx]

(np.int64(536), np.float32(0.8317791))

In [39]:
#So, we can pass the index to our knowledge base to identify the correct document
documents[idx]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [45]:
#optionally, we can sort and take the top n 

#top 5
top_5 = np.argsort(scores)[-5:]

#The elements are sorted from smallest to largest so we'll reverse it 
top_5 = top_5[::-1] 
top_5

array([536, 903, 621,   2, 501])

In [46]:
scores[top_5]

array([0.8317791 , 0.6845697 , 0.61755615, 0.60880876, 0.58479655],
      dtype=float32)

In [47]:
for idx in top_5: 
    print(scores[idx])
    print(documents[idx])
    print()
 

0.8317791
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.6845697
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

0.61755615
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 's

In [ ]:
#Start Here: https://www.youtube.com/watch?v=E7KdO3xmESg&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=23 